# Incremental CDC pipeline
Reads new SQL Server CDC events through the foreign catalog, past the last processed LSN, and applies them to Delta:

- `customer_changes`: every change event, the audit trail
- `customers_current`: a mirror of the source, maintained with `MERGE INTO` (insert, update, delete)
- `customers_history`: SCD type 2, one row per version with `valid_from` / `valid_to`

Events are ordered by LSN, so several changes of one row in a batch, or late-arriving events, are applied in commit order. The watermark moves only after all targets are updated.

In [ ]:
dbutils.widgets.text("foreign_catalog", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

from pyspark.sql import Window
from pyspark.sql import functions as F

fc = dbutils.widgets.get("foreign_catalog")
target = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
SOURCE = "sales.customers"
COLUMNS = ["customer_id", "name", "segment", "country", "email", "credit_limit"]

spark.sql(f"CREATE TABLE IF NOT EXISTS {target}.cdc_watermark (source STRING, last_lsn STRING, updated_at TIMESTAMP)")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}.customer_changes (
    customer_id INT, name STRING, segment STRING, country STRING, email STRING, credit_limit DECIMAL(12,2),
    op STRING, lsn STRING, seqval STRING, change_ts TIMESTAMP, batch_id STRING, _processed_at TIMESTAMP)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}.customers_current (
    customer_id INT, name STRING, segment STRING, country STRING, email STRING, credit_limit DECIMAL(12,2),
    last_change_ts TIMESTAMP, lsn STRING)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}.customers_history (
    customer_id INT, name STRING, segment STRING, country STRING, email STRING, credit_limit DECIMAL(12,2),
    valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN, end_reason STRING, lsn STRING, seqval STRING)""")

last = spark.sql(f"SELECT last_lsn FROM {target}.cdc_watermark WHERE source = '{SOURCE}'").collect()
last_lsn = last[0][0] if last else None
print("last processed LSN:", last_lsn)

## Read new change events
Operation codes: 1 delete, 2 insert, 4 update (after image). The before image of an update (3) is not needed.

In [ ]:
import uuid

ct = spark.table(f"{fc}.cdc.sales_customers_ct").where(F.col("`__$operation`").isin(1, 2, 4))
if last_lsn:
    ct = ct.where(F.col("`__$start_lsn`") > F.expr(f"X'{last_lsn}'"))
lsn_time = spark.table(f"{fc}.cdc.lsn_time_mapping").select(F.col("start_lsn"), F.col("tran_end_time").alias("change_ts"))

batch_id = uuid.uuid4().hex[:8]
changes = (
    ct.join(lsn_time, ct["`__$start_lsn`"] == lsn_time["start_lsn"])
    .select(
        *COLUMNS,
        F.element_at(F.array(F.lit("D"), F.lit("I"), F.lit("B"), F.lit("U")), F.col("`__$operation`")).alias("op"),
        F.hex("`__$start_lsn`").alias("lsn"),
        F.hex("`__$seqval`").alias("seqval"),
        "change_ts",
        F.lit(batch_id).alias("batch_id"),
        F.current_timestamp().alias("_processed_at"),
    )
)
changes.write.mode("append").saveAsTable(f"{target}.customer_changes")
batch = spark.table(f"{target}.customer_changes").where(F.col("batch_id") == batch_id)
batch.createOrReplaceTempView("batch")

summary = batch.groupBy("op").count().orderBy("op")
display(summary)
has_changes = not batch.isEmpty()

## Mirror: `MERGE INTO customers_current`
Only the last event per key matters for the current state.

In [ ]:
if has_changes:
    order = Window.partitionBy("customer_id").orderBy(F.col("lsn").desc(), F.col("seqval").desc())
    batch.withColumn("_rn", F.row_number().over(order)).where("_rn = 1").drop("_rn").createOrReplaceTempView("latest")
    spark.sql(f"""
        MERGE INTO {target}.customers_current t
        USING latest s ON t.customer_id = s.customer_id
        WHEN MATCHED AND s.op = 'D' THEN DELETE
        WHEN MATCHED THEN UPDATE SET
            name = s.name, segment = s.segment, country = s.country, email = s.email, credit_limit = s.credit_limit,
            last_change_ts = s.change_ts, lsn = s.lsn
        WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT
            (customer_id, name, segment, country, email, credit_limit, last_change_ts, lsn)
            VALUES (s.customer_id, s.name, s.segment, s.country, s.email, s.credit_limit, s.change_ts, s.lsn)
    """)

## History: SCD type 2
The first event of a key in the batch closes its open version. Every insert or update becomes a version that is valid until the next event of the same key; a delete only closes.

In [ ]:
if has_changes:
    by_key = Window.partitionBy("customer_id").orderBy("lsn", "seqval")
    ordered = (
        batch.withColumn("_rn", F.row_number().over(by_key))
        .withColumn("next_ts", F.lead("change_ts").over(by_key))
        .withColumn("next_op", F.lead("op").over(by_key))
    )
    ordered.where("_rn = 1").createOrReplaceTempView("first_change")
    ordered.where("op <> 'D'").createOrReplaceTempView("versions")

    spark.sql(f"""
        MERGE INTO {target}.customers_history h
        USING first_change s ON h.customer_id = s.customer_id AND h.is_current
        WHEN MATCHED THEN UPDATE SET valid_to = s.change_ts, is_current = false, end_reason = s.op
    """)
    spark.sql(f"""
        MERGE INTO {target}.customers_history h
        USING versions s ON h.customer_id = s.customer_id AND h.lsn = s.lsn AND h.seqval = s.seqval
        WHEN NOT MATCHED THEN INSERT
            (customer_id, name, segment, country, email, credit_limit, valid_from, valid_to, is_current, end_reason, lsn, seqval)
            VALUES (s.customer_id, s.name, s.segment, s.country, s.email, s.credit_limit, s.change_ts, s.next_ts,
                    s.next_ts IS NULL, s.next_op, s.lsn, s.seqval)
    """)

    new_lsn = batch.agg(F.max("lsn")).first()[0]
    spark.sql(f"""
        MERGE INTO {target}.cdc_watermark w
        USING (SELECT '{SOURCE}' AS source, '{new_lsn}' AS last_lsn) s ON w.source = s.source
        WHEN MATCHED THEN UPDATE SET last_lsn = s.last_lsn, updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (source, last_lsn, updated_at) VALUES (s.source, s.last_lsn, current_timestamp())
    """)
    print("watermark moved to", new_lsn)

## Validation
The mirror must equal the source table read through federation, and every key must have exactly one open version unless it was deleted.

In [ ]:
source = spark.table(f"{fc}.sales.customers").select(*COLUMNS)
mirror = spark.table(f"{target}.customers_current").select(*COLUMNS)
missing, extra = source.exceptAll(mirror).count(), mirror.exceptAll(source).count()

history = spark.table(f"{target}.customers_history")
open_versions = history.where("is_current").groupBy("customer_id").count()
bad_open = open_versions.where("count > 1").count()
history_vs_mirror = history.where("is_current").select(*COLUMNS).exceptAll(mirror).count()

print(f"source rows: {source.count()}, mirror rows: {mirror.count()}, missing: {missing}, extra: {extra}")
print(f"keys with several open versions: {bad_open}, open versions not in mirror: {history_vs_mirror}")
assert missing == 0 and extra == 0 and bad_open == 0 and history_vs_mirror == 0

## History of the changed customers

In [ ]:
changed = [r[0] for r in batch.select("customer_id").distinct().limit(20).collect()] if has_changes else []
if changed and len(changed) < 50:
    display(history.where(F.col("customer_id").isin(changed)).orderBy("customer_id", "valid_from"))

## Delta Change Data Feed downstream
The mirror has Change Data Feed on, so consumers can read its changes incrementally as well.

In [ ]:
version = spark.sql(f"DESCRIBE HISTORY {target}.customers_current LIMIT 1").first()["version"]
display(spark.sql(f"SELECT _change_type, count(*) AS rows FROM table_changes('{target}.customers_current', {version}) GROUP BY ALL"))